In [20]:
import random
from pathlib import Path

import numpy as np
import polars as pl
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
from PIL import Image
from sklearn.metrics import auc, average_precision_score, confusion_matrix, roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import Compose, RandomHorizontalFlip, RandomRotation, Resize, ToTensor
from tqdm import tqdm

In [2]:
SEED = 492
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
g = torch.Generator()
g.manual_seed(SEED)

In [3]:
data_path = Path("../../data/ISIC")
image_path = data_path / "ISIC_2024_Training_Input"
ground_truth_path = data_path / "ISIC_2024_Training_GroundTruth.csv"
metadata_path = data_path / "metadata.csv"
model_path = Path("../../models")

In [4]:
ground_truth_df = pl.read_csv(ground_truth_path)
metadata_df = pl.read_csv(metadata_path)

df = ground_truth_df.join(metadata_df, on="isic_id", how="inner").with_columns(
    (pl.lit(str(image_path)) + "/" + pl.col("isic_id").cast(pl.Utf8) + ".jpg").alias("image_path")
)

In [5]:
patient_stats = df.group_by("patient_id").agg(pl.col("malignant").max().alias("has_malignant"))

train_p, val_p = train_test_split(
    patient_stats, test_size=0.1, random_state=SEED, stratify=patient_stats["has_malignant"]
)

train_ids = train_p["patient_id"].to_list()
val_ids = val_p["patient_id"].to_list()

train_df = df.filter(pl.col("patient_id").is_in(train_ids))
val_df = df.filter(pl.col("patient_id").is_in(val_ids))

In [6]:
train_malignant = train_df.filter(pl.col("malignant") == 1)
train_benign = train_df.filter(pl.col("malignant") == 0)

capped_benign = (
    train_benign
    .sample(fraction=1.0, shuffle=True, seed=SEED)
    .group_by("patient_id")
    .head(10)
    .select(train_df.columns)
)

final_train_df = pl.concat([train_malignant, capped_benign])

print(f"Train Malignant: {train_malignant.height}")
print(f"Train Benign (capped): {capped_benign.height}")
print(f"Total Train: {final_train_df.height}")

Train Malignant: 347
Train Benign (capped): 9277
Total Train: 9624


In [7]:
tab_categorical = ["sex", "anatom_site_general"]
tab_numerical = ["age_approx", "clin_size_long_diam_mm", "tbp_lv_areaMM2", "tbp_lv_eccentricity"]

median_age = final_train_df["age_approx"].median()
mode_sex = final_train_df["sex"].drop_nulls().mode()[0]

In [8]:
final_train_df = final_train_df.with_columns(
    pl.col("age_approx").fill_null(median_age),
    pl.col("sex").fill_null(mode_sex),
    pl.col("anatom_site_general").fill_null("unknown")
)

val_df = val_df.with_columns(
    pl.col("age_approx").fill_null(median_age),
    pl.col("sex").fill_null(mode_sex),
    pl.col("anatom_site_general").fill_null("unknown")
)

In [9]:
exprs = []
for c in tab_numerical:
    exprs.append(pl.col(c).mean().alias(f"{c}_mean"))
    exprs.append(pl.col(c).std().alias(f"{c}_std"))

num_stats = final_train_df.select(exprs)

for col in tab_numerical:
    mean = num_stats.item(0, f"{col}_mean")
    std = num_stats.item(0, f"{col}_std")
    final_train_df = final_train_df.with_columns(((pl.col(col) - mean) / std).alias(col))
    val_df = val_df.with_columns(((pl.col(col) - mean) / std).alias(col))

In [10]:
tab_features = list(tab_numerical)

for col in tab_categorical:
    categories = final_train_df[col].unique().to_list()
    for cat in categories:
        col_name = f"{col}_{cat}"
        final_train_df = final_train_df.with_columns((pl.col(col) == cat).cast(pl.Int8).alias(col_name))
        val_df = val_df.with_columns((pl.col(col) == cat).cast(pl.Int8).alias(col_name))
        tab_features.append(col_name)

final_train_df = final_train_df.drop(tab_categorical)
val_df = val_df.drop(tab_categorical)

In [11]:
class ISICMultimodalDataset(Dataset):
    def __init__(self, dataframe: pl.DataFrame, tabular_features: list[str], image_transform):
        self.df = dataframe
        self.tab_features = tabular_features
        self.image_transform = image_transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.row(idx, named=True)

        img = Image.open(row["image_path"]).convert("RGB")
        img = self.image_transform(img)

        tab_data = torch.tensor([row[feat] for feat in self.tab_features], dtype=torch.float32)
        label = torch.tensor(row["malignant"], dtype=torch.float32)

        return img, tab_data, label

In [12]:
train_transform = Compose([
    Resize((224, 224)),
    RandomHorizontalFlip(p=0.5),
    RandomRotation(15),
    ToTensor()
])

val_transform = Compose([
    Resize((224, 224)),
    ToTensor()
])

In [13]:
train_dataset = ISICMultimodalDataset(final_train_df, tab_features, train_transform)
val_dataset = ISICMultimodalDataset(val_df, tab_features, val_transform)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, generator=g)
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False)

In [14]:
class MultimodalModel(nn.Module):
    def __init__(self, tab_dim):
        super().__init__()
        self.image_encoder = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        self.image_encoder.fc = nn.Identity()

        self.tab_encoder = nn.Sequential(
            nn.Linear(tab_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(64, 32)
        )

        self.fusion_head = nn.Sequential(
            nn.Linear(2048 + 32, 512),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(512, 1)
        )

    def forward(self, image, tab):
        img_feat = self.image_encoder(image)
        tab_feat = self.tab_encoder(tab)
        combined = torch.cat([img_feat, tab_feat], dim=1)
        return self.fusion_head(combined)

In [15]:
device = torch.device("cuda")
model = MultimodalModel(tab_dim=len(tab_features)).to(device)

train_malignant_count = final_train_df.filter(pl.col("malignant") == 1).height
train_benign_count = final_train_df.filter(pl.col("malignant") == 0).height

pos_weight = torch.tensor([train_benign_count / train_malignant_count]).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.AdamW(model.parameters(), lr=1e-5, weight_decay=3e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.1, patience=3)
scaler = torch.amp.GradScaler("cuda")

In [16]:
best_ap = 0.0
best_recall_at_spec = 0.0
early_stopping_counter = 0
early_stopping_patience = 5
num_epochs = 30

model_path.mkdir(parents=True, exist_ok=True)

for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0
    train_loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Train]", leave=False)
    for images, tabs, labels in train_loop:
        images, tabs, labels = images.to(device), tabs.to(device), labels.to(device)
        optimizer.zero_grad()
        with torch.amp.autocast(device_type="cuda"):
            outputs = model(images, tabs)
            loss = criterion(outputs, labels.unsqueeze(1))
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        train_loss += loss.item()
        train_loop.set_postfix(loss=loss.item())

    avg_train_loss = train_loss / len(train_loader)

    model.eval()
    val_loss = 0.0
    all_preds = []
    all_labels = []
    val_loop = tqdm(val_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Val]", leave=False)
    with torch.no_grad():
        for images, tabs, labels in val_loop:
            images, tabs, labels = images.to(device), tabs.to(device), labels.to(device)
            with torch.amp.autocast(device_type="cuda"):
                outputs = model(images, tabs)
                loss = criterion(outputs, labels.unsqueeze(1))
            val_loss += loss.item()
            all_preds.extend(torch.sigmoid(outputs).squeeze(1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_val_loss = val_loss / len(val_loader)

    ap = average_precision_score(all_labels, all_preds)
    fpr, tpr, thresholds = roc_curve(all_labels, all_preds)

    valid_indices = np.where(fpr <= 0.10)[0]
    recall_at_spec = tpr[valid_indices[-1]]

    scheduler.step(ap)

    if ap > best_ap:
        best_ap = ap
        early_stopping_counter = 0
    else:
        early_stopping_counter += 1

    if recall_at_spec > best_recall_at_spec:
        best_recall_at_spec = recall_at_spec
        torch.save(model.state_dict(), model_path / "best_multimodal_model.pt")

    print(f"Epoch {epoch+1}/{num_epochs} | "
          f"Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | "
          f"AP: {ap:.4f} | Recall@90Spec: {recall_at_spec:.4f}")

    if early_stopping_counter >= early_stopping_patience:
        print(f"Early stopping triggered at epoch {epoch+1}")
        break

Epoch 1/30 | Train Loss: 1.2991 | Val Loss: 0.6581 | AP: 0.0277 | Recall@90Spec: 0.3696


Epoch 2/30 | Train Loss: 1.2093 | Val Loss: 0.6228 | AP: 0.0152 | Recall@90Spec: 0.5000


Epoch 3/30 | Train Loss: 1.0189 | Val Loss: 0.6067 | AP: 0.0375 | Recall@90Spec: 0.6304


Epoch 4/30 | Train Loss: 0.8127 | Val Loss: 0.5166 | AP: 0.0328 | Recall@90Spec: 0.7174


Epoch 5/30 | Train Loss: 0.7213 | Val Loss: 0.4503 | AP: 0.0430 | Recall@90Spec: 0.7174


Epoch 6/30 | Train Loss: 0.6366 | Val Loss: 0.4625 | AP: 0.0306 | Recall@90Spec: 0.7391


Epoch 7/30 | Train Loss: 0.5655 | Val Loss: 0.3639 | AP: 0.0283 | Recall@90Spec: 0.7609


Epoch 8/30 | Train Loss: 0.5319 | Val Loss: 0.3257 | AP: 0.0295 | Recall@90Spec: 0.8043


Epoch 9/30 | Train Loss: 0.4726 | Val Loss: 0.3383 | AP: 0.0221 | Recall@90Spec: 0.7826


Epoch 10/30 | Train Loss: 0.4261 | Val Loss: 0.3263 | AP: 0.0258 | Recall@90Spec: 0.7609
Early stopping triggered at epoch 10


In [17]:
model.load_state_dict(torch.load(model_path / "best_multimodal_model.pt", weights_only=True))
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for images, tabs, labels in tqdm(val_loader, desc="Evaluating Best Model"):
        images, tabs, labels = images.to(device), tabs.to(device), labels.to(device)
        with torch.amp.autocast(device_type="cuda"):
            outputs = model(images, tabs)
        all_preds.extend(torch.sigmoid(outputs).squeeze(1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

auc_roc = roc_auc_score(all_labels, all_preds)
ap = average_precision_score(all_labels, all_preds)

fpr, tpr, thresholds = roc_curve(all_labels, all_preds)
valid_idx_90 = np.where(fpr <= 0.10)[0]
threshold_90_spec = thresholds[valid_idx_90[-1]]
recall_90_spec = tpr[valid_idx_90[-1]]

valid_idx_95 = np.where(fpr <= 0.05)[0]
threshold_95_spec = thresholds[valid_idx_95[-1]]
recall_95_spec = tpr[valid_idx_95[-1]]

preds_binary = (all_preds >= threshold_90_spec).astype(int)
tn, fp, fn, tp = confusion_matrix(all_labels, preds_binary).ravel()

print(f"AUC-ROC: {auc_roc:.4f}")
print(f"Average Precision: {ap:.4f}")
print(f"--- @ 90% Spec ---")
print(f"Threshold: {threshold_90_spec:.4f} | Recall: {recall_90_spec:.4f}")
print(f"TP: {tp} | FP: {fp} | TN: {tn} | FN: {fn}")
print(f"--- @ 95% Spec ---")
print(f"Threshold: {threshold_95_spec:.4f} | Recall: {recall_95_spec:.4f}")

Evaluating Best Model: 100%|██████████| 341/341 [01:09<00:00,  4.92it/s]


AUC-ROC: 0.8835
Average Precision: 0.0295
--- @ 90% Spec ---
Threshold: 0.5361 | Recall: 0.8043
TP: 37 | FP: 4355 | TN: 39235 | FN: 9
--- @ 95% Spec ---
Threshold: 0.7388 | Recall: 0.6739


In [21]:
def p_auc_tpr(v_gt, v_pred, min_tpr=0.80):
    v_gt_flipped = abs(np.asarray(v_gt) - 1)
    v_pred_flipped = abs(np.asarray(v_pred) - 1)
    max_fpr = abs(1 - min_tpr)

    fpr, tpr, _ = roc_curve(v_gt_flipped, v_pred_flipped)

    stop = np.searchsorted(fpr, max_fpr, "right")
    x_interp = [fpr[stop-1], fpr[stop]]
    y_interp = [tpr[stop-1], tpr[stop]]

    tpr_adj = np.append(tpr[:stop], np.interp(max_fpr, x_interp, y_interp))
    fpr_adj = np.append(fpr[:stop], max_fpr)

    return auc(fpr_adj, tpr_adj)

isic_pauc = p_auc_tpr(all_labels, all_preds, min_tpr=0.80)
print(f"ISIC 2024 Official Metric (pAUC > 80% TPR): {isic_pauc:.5f}")

ISIC 2024 Official Metric (pAUC > 80% TPR): 0.10240
